In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# =========================
# PARAMETERS
# =========================
tickers = ['GEV', 'AGQ', 'VGSH',]
market_ticker = '^GSPC'

end_date = datetime.today()
start_date = end_date - timedelta(days=3 * 365)

# =========================
# PRICE DATA
# =========================
prices = yf.download(
    tickers + [market_ticker],
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=False
)['Close'].dropna()

returns = prices.pct_change().dropna()

asset_returns = returns[tickers]
market_returns = returns[market_ticker]

# =========================
# RISK-FREE RATE (ROBUST)
# =========================
annual_rf = 0.05        # 5% annual RF proxy
rf_daily = annual_rf / 252

rf_daily = pd.Series(
    rf_daily,
    index=asset_returns.index,
    name='RF'
)

# =========================
# EXCESS RETURNS
# =========================
excess_returns = asset_returns.subtract(rf_daily, axis=0)
market_excess = market_returns.subtract(rf_daily, axis=0)
market_excess.name = 'MKT'

aligned = excess_returns.join(market_excess, how='inner')

# =========================
# RISK PREMIA & SHARPE
# =========================
daily_premia = excess_returns.mean()
annual_premia = daily_premia * 252

annual_vol = excess_returns.std() * np.sqrt(252)
sharpe_ratio = annual_premia / annual_vol

# =========================
# BETA (CAPM)
# =========================
betas = {}
var_mkt = np.var(aligned['MKT'])

for asset in tickers:
    cov = np.cov(aligned[asset], aligned['MKT'])[0, 1]
    betas[asset] = cov / var_mkt

betas = pd.Series(betas)

# =========================
# RESULTS
# =========================
results = pd.DataFrame({
    'Daily Risk Premium': daily_premia,
    'Annual Risk Premium': annual_premia,
    'Annual Volatility': annual_vol,
    'Sharpe Ratio': sharpe_ratio,
    'Market Beta': betas
})

print(results.round(4))





      Daily Risk Premium  Annual Risk Premium  Annual Volatility  \
GEV               0.0043               1.0863             0.5400   
AGQ               0.0043               1.0759             0.6477   
VGSH             -0.0000              -0.0010             0.0158   

      Sharpe Ratio  Market Beta  
GEV         2.0117       1.8708  
AGQ         1.6610       1.1223  
VGSH       -0.0663      -0.0191  
